# exp_002 — Quantization with llama.cpp GGUF

This notebook loads the strongest available measured summary. By default it displays the committed v002 pilot; set `EXP002_PHASE=full` to require a complete matrix and fail closed when `summary.csv` is absent. It never invents missing variant measurements.

Capability outcomes and systems costs are separate. Speed fields are stream-derived proxies; native prefill/decode counters are unavailable. RSS is sampled within the process and is not a definitive cross-variant memory comparison when variants are loaded sequentially. The pilot is one greedy run per independent task; repeated greedy prompts are not independent accuracy samples.

In [ ]:
import json
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / 'src').is_dir() and (candidate / 'experiments').is_dir())
EXPERIMENT_ROOT = ROOT / 'experiments/exp_002-quantization_llama_cpp_gguf'
RESULTS_DIR = Path(os.environ.get('EXP002_RESULTS_DIR', ROOT / 'experiments/exp_002-quantization_llama_cpp_gguf/results'))
SUMMARY_PATH = RESULTS_DIR / 'processed/summary.csv'
PILOT_SUMMARY_PATH = RESULTS_DIR / 'processed/pilot-v002-summary.csv'
MANIFEST_PATH = RESULTS_DIR / 'manifest.json'
PHASE = os.environ.get('EXP002_PHASE', 'pilot')
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'resolved manifest is required: {MANIFEST_PATH}')
if PHASE == 'full':
    if not SUMMARY_PATH.is_file():
        raise FileNotFoundError(f'complete measured summary is required: {SUMMARY_PATH}')
    selected_summary_path = SUMMARY_PATH
elif PHASE == 'pilot':
    if not PILOT_SUMMARY_PATH.is_file():
        raise FileNotFoundError(f'pilot summary is required: {PILOT_SUMMARY_PATH}')
    selected_summary_path = PILOT_SUMMARY_PATH
else:
    raise ValueError("EXP002_PHASE must be 'pilot' or 'full'")
sys.path.insert(0, str(ROOT / 'src'))
from llm_lab.analysis.quantization import recommend_baseline, tradeoff_rows
from llm_lab.quantization import QuantizationManifest
manifest_record = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
manifest = QuantizationManifest.from_record(manifest_record)
summaries = pd.read_csv(selected_summary_path)
EXPECTED_SCORER = 'calibrated.v1'
if 'scorer_version' not in summaries.columns or set(summaries['scorer_version'].dropna().astype(str)) != {EXPECTED_SCORER}:
    raise ValueError("summary scorer policy must be 'calibrated.v1'; legacy summaries are not valid")
required_columns = {'condition_id', 'attempted_n', 'scored_n', 'correct_n', 'scored_accuracy', 'answer_bearing_correct_n', 'answer_bearing_scored_n', 'answer_bearing_accuracy', 'format_valid_n', 'format_scored_n', 'format_validity', 'end_to_end_success', 'failure_rate', 'median_stream_ttft_s', 'median_prompt_throughput_proxy_tok_s', 'median_post_first_chunk_output_tok_s', 'median_peak_memory_bytes'}
missing_columns = required_columns - set(summaries.columns)
if missing_columns:
    raise ValueError(f'summary is missing required columns: {sorted(missing_columns)}')
print({'phase': PHASE, 'summary': str(selected_summary_path), 'rows': len(summaries), 'measured_conditions': sorted(set(summaries.get('variant_condition_id', summaries['condition_id']).astype(str)))})

In [ ]:
def _measured_rows(summary_frame):
    rows = []
    for condition_id, group in summary_frame.groupby(summary_frame.get('variant_condition_id', summary_frame['condition_id']).astype(str)):
        attempted = int(group['attempted_n'].sum())
        scored = int(group['scored_n'].sum())
        correct = int(group['correct_n'].sum())
        failures = int(group['failure_n'].sum()) if 'failure_n' in group else attempted - scored
        variant = next((item for item in manifest.variants if item.condition_id == condition_id), None)
        if variant is None:
            continue
        answer_bearing_scored = int(group['answer_bearing_scored_n'].sum())
        answer_bearing_correct = int(group['answer_bearing_correct_n'].sum())
        format_scored = int(group['format_scored_n'].sum())
        format_valid = int(group['format_valid_n'].sum())
        rows.append({'condition_id': condition_id, 'label': variant.label, 'artifact_size_bytes': variant.artifact.artifact_size_bytes, 'median_peak_memory_bytes': float(group['median_peak_memory_bytes'].median()), 'median_stream_ttft_s': float(group['median_stream_ttft_s'].median()), 'median_prompt_throughput_proxy_tok_s': float(group['median_prompt_throughput_proxy_tok_s'].median()), 'median_post_first_chunk_output_tok_s': float(group['median_post_first_chunk_output_tok_s'].median()), 'attempted_n': attempted, 'scored_n': scored, 'correct_n': correct, 'failure_n': failures, 'scored_accuracy': correct / scored if scored else None, 'answer_bearing_correct_n': answer_bearing_correct, 'answer_bearing_scored_n': answer_bearing_scored, 'answer_bearing_accuracy': answer_bearing_correct / answer_bearing_scored if answer_bearing_scored else None, 'format_valid_n': format_valid, 'format_scored_n': format_scored, 'format_validity': format_valid / format_scored if format_scored else None, 'end_to_end_success': correct / attempted if attempted else None, 'failure_rate': failures / attempted if attempted else None})
    return rows

if PHASE == 'full':
    rows = tradeoff_rows(summaries.to_dict('records'), manifest, require_complete=True)
else:
    rows = _measured_rows(summaries)
frame = pd.DataFrame(rows)
capability_frame = frame[['condition_id', 'attempted_n', 'scored_n', 'correct_n', 'scored_accuracy', 'answer_bearing_correct_n', 'answer_bearing_scored_n', 'answer_bearing_accuracy', 'format_valid_n', 'format_scored_n', 'format_validity', 'end_to_end_success', 'failure_rate']].sort_values('condition_id')
systems_cost_frame = frame[['condition_id', 'artifact_size_bytes', 'median_peak_memory_bytes', 'median_stream_ttft_s', 'median_prompt_throughput_proxy_tok_s', 'median_post_first_chunk_output_tok_s']].sort_values('condition_id')
accuracy_vs_memory = capability_frame.merge(systems_cost_frame[['condition_id', 'artifact_size_bytes']], on='condition_id')
speed_vs_memory = systems_cost_frame[['condition_id', 'artifact_size_bytes', 'median_stream_ttft_s', 'median_prompt_throughput_proxy_tok_s', 'median_post_first_chunk_output_tok_s']]
display(capability_frame)
display(systems_cost_frame)
if PHASE == 'pilot':
    print('This is a Q8_0-only pilot: no cross-quantization recommendation is valid yet.')

In [ ]:
RESULTS_DIR.joinpath('figures').mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(frame['artifact_size_bytes'] / 1e9, frame['end_to_end_success'], s=80)
for _, row in frame.iterrows():
    ax.annotate(row['label'], (row['artifact_size_bytes'] / 1e9, row['end_to_end_success']), xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('GGUF artifact size (GB)')
ax.set_ylabel('End-to-end success (correct / attempted)')
ax.set_ylim(0, 1.05)
ax.set_title(f'exp_002 capability vs artifact size ({PHASE})')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'figures' / 'accuracy-vs-memory.png', dpi=160, bbox_inches='tight')
plt.show()

if 'target_context_tokens' in summaries.columns:
    timing = summaries.groupby('target_context_tokens', as_index=False)[['median_stream_ttft_s', 'median_prompt_throughput_proxy_tok_s', 'median_post_first_chunk_output_tok_s']].median()
    display(timing)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(timing['target_context_tokens'], timing['median_stream_ttft_s'], marker='o', label='stream TTFT')
    axes[0].set_ylabel('stream TTFT (s)')
    axes[1].plot(timing['target_context_tokens'], timing['median_prompt_throughput_proxy_tok_s'], marker='o', label='prompt proxy')
    axes[1].plot(timing['target_context_tokens'], timing['median_post_first_chunk_output_tok_s'], marker='o', label='post-first-chunk output')
    axes[1].set_ylabel('tokens / second')
    for axis in axes:
        axis.set_xlabel('input context tokens')
        axis.legend()
    fig.suptitle('Stream-derived timing proxies; native kernel counters unavailable')
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'figures' / 'speed-vs-memory.png', dpi=160, bbox_inches='tight')
    plt.show()

In [ ]:
if PHASE == 'full':
    recommendation = recommend_baseline(rows, accuracy_tolerance=0.02)
    print({'recommended_condition': recommendation['condition_id'], 'recommended_label': recommendation['label'], 'end_to_end_success': recommendation['end_to_end_success'], 'artifact_size_bytes': recommendation['artifact_size_bytes'], 'accuracy_tolerance': recommendation['accuracy_tolerance']})
else:
    print('Recommendation withheld: complete Q8/Q6/Q5/Q4 capability coverage is required.')